In [1]:
from pathlib import Path
import pandas as pd

In [2]:
path = Path(r"D:\yield_curve_forecast\data\raw")

#stores data in a dictionary with file name as key and the filedata as the value.
data = {
    f.stem: pd.read_csv(f)
    for f in path.glob("*.csv")
}

#the columns present in the data.
print(data.keys())

dict_keys(['CPIAUCSL', 'CPILFESL', 'DCOILWTICO', 'DGS1', 'DGS10', 'DGS2', 'DGS30', 'DGS3MO', 'DGS5', 'FEDFUNDS', 'GDPC1', 'HOUST', 'ICSA', 'INDPRO', 'RSAFS', 'UMCSENT', 'UNRATE', 'VIXCLS'])


In [3]:
##examine the data!
for name, df in data.items():
    print(name)
    print(df.head())

CPIAUCSL
  observation_date  CPIAUCSL
0       1992-01-01     138.3
1       1992-02-01     138.6
2       1992-03-01     139.1
3       1992-04-01     139.4
4       1992-05-01     139.7
CPILFESL
  observation_date  CPILFESL
0       1992-01-01     145.1
1       1992-02-01     145.4
2       1992-03-01     145.9
3       1992-04-01     146.3
4       1992-05-01     146.8
DCOILWTICO
  observation_date  DCOILWTICO
0       1992-01-02       19.43
1       1992-01-03       19.22
2       1992-01-06       19.24
3       1992-01-07       18.72
4       1992-01-08       17.95
DGS1
  observation_date  DGS1
0       1992-01-02  4.13
1       1992-01-03  4.16
2       1992-01-06  4.12
3       1992-01-07  4.03
4       1992-01-08  4.00
DGS10
  observation_date  DGS10
0       1992-01-02   6.78
1       1992-01-03   6.85
2       1992-01-06   6.82
3       1992-01-07   6.76
4       1992-01-08   6.77
DGS2
  observation_date  DGS2
0       1992-01-02  4.77
1       1992-01-03  4.80
2       1992-01-06  4.76
3       1992-01

In [4]:
#checking the datatypes of the columns
for name, df in data.items():
    print(name)
    print(df.dtypes)

CPIAUCSL
observation_date        str
CPIAUCSL            float64
dtype: object
CPILFESL
observation_date        str
CPILFESL            float64
dtype: object
DCOILWTICO
observation_date        str
DCOILWTICO          float64
dtype: object
DGS1
observation_date        str
DGS1                float64
dtype: object
DGS10
observation_date        str
DGS10               float64
dtype: object
DGS2
observation_date        str
DGS2                float64
dtype: object
DGS30
observation_date        str
DGS30               float64
dtype: object
DGS3MO
observation_date        str
DGS3MO              float64
dtype: object
DGS5
observation_date        str
DGS5                float64
dtype: object
FEDFUNDS
observation_date        str
FEDFUNDS            float64
dtype: object
GDPC1
observation_date        str
GDPC1               float64
dtype: object
HOUST
observation_date      str
HOUST               int64
dtype: object
ICSA
observation_date      str
ICSA                int64
dtype: object
INDPRO
ob

In [5]:
for name, df in data.items():
    df["observation_date"] = pd.to_datetime(df["observation_date"]) #converting observation date from string to datetime format.
    
busi_col = []

## checking the frequency of observations for each of the columns!
for name, df in data.items():
    freq = pd.infer_freq(df["observation_date"])
    print(f"{name}: {freq}")
    if freq == "B":
        busi_col.append(name)

    # D = daily
    # B = business day: observations exist only on weekdays.
    # W = weekly: (e.g. W-SAT: means week ends on saturday, ie.e, the timestamp of value is at each Saturday.)
    #MS = month start 
    #ME = month end
    #QS = quarter start: (e.g. QS-OCT: means that the last quarter is October.)

CPIAUCSL: MS
CPILFESL: MS
DCOILWTICO: B
DGS1: B
DGS10: B
DGS2: B
DGS30: B
DGS3MO: B
DGS5: B
FEDFUNDS: MS
GDPC1: QS-OCT
HOUST: MS
ICSA: W-SAT
INDPRO: MS
RSAFS: MS
UMCSENT: MS
UNRATE: MS
VIXCLS: B


In [6]:
## find the missing value columns only!

dfna = {}
for name, df in data.items():
    if df.iloc[:,1].isna().sum() != 0:
        print(name, df.iloc[:,1].isna().sum() )

        df_na = df[df.iloc[:,1].isna()]

        dfna[name] = df_na
     #df[:,1] does not work properly, df[name] can also be used.

DCOILWTICO 329
DGS1 361
DGS10 361
DGS2 361
DGS30 361
DGS3MO 361
DGS5 361
VIXCLS 282


In [7]:
## drop the NaN values for columns which have frequency: B = Business days.
drop_nans = []
for name, df in data.items():
    if (df.iloc[:,1].isna().sum() != 0) & (pd.infer_freq(df["observation_date"])=="B"):
        drop_nans.append(name)

print(drop_nans)

for name, df in data.items():
    if name in drop_nans:
        df.dropna(inplace= True)

['DCOILWTICO', 'DGS1', 'DGS10', 'DGS2', 'DGS30', 'DGS3MO', 'DGS5', 'VIXCLS']


In [8]:
## make the frequency for the data's columns to be monthly.

# Business to Monthly or Weekly to Monthly:
for name, df in data.items():
    df = df.set_index("observation_date")

    if name in busi_col:
        df = df.resample("MS").last()   # month-end snapshot for yields/VIX/oil
    elif name == "ICSA":
        df = df.resample("MS").mean()   # weekly claims: flow, so average the month

    data[name] = df.reset_index()

#Quarterly to Monhtly:
dates = data["DGS1"]["observation_date"]
df = data["GDPC1"].set_index("observation_date").reindex(dates, method= "ffill")
data["GDPC1"] = df.reset_index()

for name, df in data.items():
    print(name)
    print(df.head())

CPIAUCSL
  observation_date  CPIAUCSL
0       1992-01-01     138.3
1       1992-02-01     138.6
2       1992-03-01     139.1
3       1992-04-01     139.4
4       1992-05-01     139.7
CPILFESL
  observation_date  CPILFESL
0       1992-01-01     145.1
1       1992-02-01     145.4
2       1992-03-01     145.9
3       1992-04-01     146.3
4       1992-05-01     146.8
DCOILWTICO
  observation_date  DCOILWTICO
0       1992-01-01       18.93
1       1992-02-01       18.69
2       1992-03-01       19.49
3       1992-04-01       20.88
4       1992-05-01       22.13
DGS1
  observation_date  DGS1
0       1992-01-01  4.23
1       1992-02-01  4.35
2       1992-03-01  4.54
3       1992-04-01  4.40
4       1992-05-01  4.24
DGS10
  observation_date  DGS10
0       1992-01-01   7.31
1       1992-02-01   7.27
2       1992-03-01   7.54
3       1992-04-01   7.61
4       1992-05-01   7.33
DGS2
  observation_date  DGS2
0       1992-01-01  5.11
1       1992-02-01  5.27
2       1992-03-01  5.60
3       1992-04

In [9]:
new_df = pd.DataFrame(dates)

for name, df in data.items():
    new_df = new_df.merge(
        df[["observation_date", name]],
        on="observation_date",
        how="left"
    )

print(new_df.head())

  observation_date  CPIAUCSL  CPILFESL  DCOILWTICO  DGS1  DGS10  DGS2  DGS30  \
0       1992-01-01     138.3     145.1       18.93  4.23   7.31  5.11   7.77   
1       1992-02-01     138.6     145.4       18.69  4.35   7.27  5.27   7.80   
2       1992-03-01     139.1     145.9       19.49  4.54   7.54  5.60   7.96   
3       1992-04-01     139.4     146.3       20.88  4.40   7.61  5.46   8.06   
4       1992-05-01     139.7     146.8       22.13  4.24   7.33  5.19   7.84   

   DGS3MO  DGS5  FEDFUNDS      GDPC1  HOUST      ICSA   INDPRO   RSAFS  \
0    3.94  6.44      4.03  10236.435   1176  439000.0  61.4616  159177   
1    4.03  6.58      4.06  10236.435   1250  442200.0  61.8955  159189   
2    4.15  6.94      3.98  10236.435   1297  429500.0  62.4232  158647   
3    3.79  6.91      3.73  10347.429   1099  418250.0  62.8970  159921   
4    3.79  6.61      3.82  10347.429   1214  417400.0  63.1040  160471   

   UMCSENT  UNRATE  VIXCLS  
0     67.5     7.3   17.40  
1     68.8     7

In [10]:
print(new_df.isna().sum())
print("Duplicate dates:", new_df["observation_date"].duplicated().sum())

observation_date    0
CPIAUCSL            0
CPILFESL            0
DCOILWTICO          0
DGS1                0
DGS10               0
DGS2                0
DGS30               0
DGS3MO              0
DGS5                0
FEDFUNDS            0
GDPC1               0
HOUST               0
ICSA                0
INDPRO              0
RSAFS               0
UMCSENT             0
UNRATE              0
VIXCLS              0
dtype: int64
Duplicate dates: 0


In [11]:
## Saving the processed data.

output_path = Path(r"D:\yield_curve_forecast\data\processed")
new_df.to_csv(output_path / "yield_curve_data.csv", index=False)
new_df.to_parquet(output_path / "yield_curve_data.parquet")